In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
init_load_flag = int(dbutils.widgets.get("init_load_flag"))

### **Data Reading from source**

In [0]:
df = spark.sql("select * from databricks_cata.silver.customers_silver")

In [0]:
df.display()

customer_id,email,city,state,domains,full_name
C00001,rushjeff@ryan.org,Johnsonmouth,MS,ryan.org,Emily Mooney
C00002,mccoykiara@kelly.com,Stephenfort,WY,kelly.com,Andrea Sellers
C00003,rebeccamiller@yahoo.com,South Stephenshire,LA,yahoo.com,Craig Hayes
C00004,lawrence05@campbell.info,Chrisland,ND,campbell.info,Bryan Scott
C00005,carrie45@yahoo.com,East Dennistown,RI,yahoo.com,Sean Vasquez
C00006,traceyramos@gmail.com,North Matthew,IN,gmail.com,Kevin Mccarthy
C00007,scottallen@gmail.com,Joneshaven,VA,gmail.com,Amanda Doyle
C00008,sullivanjeremy@horton-adams.com,South Nathanfurt,CT,horton-adams.com,Paul Campos
C00009,dennis03@yahoo.com,Kimberlyview,MD,yahoo.com,Mary Green
C00010,charles58@murillo.net,West Hector,OK,murillo.net,James Myers


**Remove Duplicates**

In [0]:
df = df.dropDuplicates(subset = ['customer_id'])
df.limit(10).display()

customer_id,email,city,state,domains,full_name
C00049,kristinawalsh@gmail.com,New Heatherside,IA,gmail.com,Mike Harvey
C00126,bonniewhite@smith.com,Gregoryton,WA,smith.com,Amanda King
C00169,michaelfox@hotmail.com,Michaelburgh,DC,hotmail.com,Holly Ward
C00394,thomasmontgomery@gonzalez-ross.com,North Denisemouth,NJ,gonzalez-ross.com,Anthony Kelly
C00509,dylanmclean@franco-little.org,Pachecotown,TX,franco-little.org,Andre Murray
C00929,buchananjerry@evans-allen.biz,Wellsshire,NC,evans-allen.biz,Timothy Atkinson
C01462,sflynn@yahoo.com,Justinstad,MD,yahoo.com,Melody Daniels
C01720,carolmartin@yahoo.com,North Brianview,IN,yahoo.com,Cindy Gonzalez
C01815,aperez@gmail.com,Port Michaelstad,ID,gmail.com,Laura Kramer
C00006,traceyramos@gmail.com,North Matthew,IN,gmail.com,Kevin Mccarthy


### **Assign a surrogate key column to our DataFrame**

### Surrogate key is a pseudokey that we create in our dimension so that we can easily apply joins



## **Dividing New vs Old Records**

In [0]:
if init_load_flag == 0:
    df_old = spark.sql("""
        select DimCustomerKey, customer_id, create_date, update_date 
        from databricks_cata.gold.DimCustomers
    """)

else:
    df_old = spark.sql("""
        select 0 DimCustomerKey, 0 customer_id, 0 create_date, 0 update_date
        from databricks_cata.silver.customers_silver 
        where 1 = 0
    """)

In [0]:
df_old.display()

DimCustomerKey,customer_id,create_date,update_date
1,C00001,2025-09-03T05:36:15.544Z,2025-09-03T05:36:15.544Z
2,C00002,2025-09-03T05:36:15.544Z,2025-09-03T05:36:15.544Z
3,C00003,2025-09-03T05:36:15.544Z,2025-09-03T05:36:15.544Z
4,C00004,2025-09-03T05:36:15.544Z,2025-09-03T05:36:15.544Z
5,C00005,2025-09-03T05:36:15.544Z,2025-09-03T05:36:15.544Z
6,C00006,2025-09-03T05:36:15.544Z,2025-09-03T05:36:15.544Z
7,C00007,2025-09-03T05:36:15.544Z,2025-09-03T05:36:15.544Z
8,C00008,2025-09-03T05:36:15.544Z,2025-09-03T05:36:15.544Z
9,C00009,2025-09-03T05:36:15.544Z,2025-09-03T05:36:15.544Z
10,C00010,2025-09-03T05:36:15.544Z,2025-09-03T05:36:15.544Z


**Renaming columns of df_old**

In [0]:
df_old = df_old.withColumnRenamed("DimCustomerKey", "old_DimCustomerKey").withColumnRenamed("customer_id", "old_customer_id").withColumnRenamed("create_date", "old_create_date").withColumnRenamed("update_date", "old_update_date")

In [0]:
df_old.display()

old_DimCustomerKey,old_customer_id,old_create_date,old_update_date
1,C00001,2025-09-03T05:36:15.544Z,2025-09-03T05:36:15.544Z
2,C00002,2025-09-03T05:36:15.544Z,2025-09-03T05:36:15.544Z
3,C00003,2025-09-03T05:36:15.544Z,2025-09-03T05:36:15.544Z
4,C00004,2025-09-03T05:36:15.544Z,2025-09-03T05:36:15.544Z
5,C00005,2025-09-03T05:36:15.544Z,2025-09-03T05:36:15.544Z
6,C00006,2025-09-03T05:36:15.544Z,2025-09-03T05:36:15.544Z
7,C00007,2025-09-03T05:36:15.544Z,2025-09-03T05:36:15.544Z
8,C00008,2025-09-03T05:36:15.544Z,2025-09-03T05:36:15.544Z
9,C00009,2025-09-03T05:36:15.544Z,2025-09-03T05:36:15.544Z
10,C00010,2025-09-03T05:36:15.544Z,2025-09-03T05:36:15.544Z


**Applying join with the old records**

In [0]:
df_join = df.join(df_old, df.customer_id == df_old.old_customer_id, 'left')

In [0]:
#All the values will be null as this is an initial load

df_join.display()

customer_id,email,city,state,domains,full_name,old_DimCustomerKey,old_customer_id,old_create_date,old_update_date
C00049,kristinawalsh@gmail.com,New Heatherside,IA,gmail.com,Mike Harvey,49,C00049,2025-09-03T05:36:15.544Z,2025-09-03T05:36:15.544Z
C00126,bonniewhite@smith.com,Gregoryton,WA,smith.com,Amanda King,126,C00126,2025-09-03T05:36:15.544Z,2025-09-03T05:36:15.544Z
C00169,michaelfox@hotmail.com,Michaelburgh,DC,hotmail.com,Holly Ward,169,C00169,2025-09-03T05:36:15.544Z,2025-09-03T05:36:15.544Z
C00394,thomasmontgomery@gonzalez-ross.com,North Denisemouth,NJ,gonzalez-ross.com,Anthony Kelly,394,C00394,2025-09-03T05:36:15.544Z,2025-09-03T05:36:15.544Z
C00509,dylanmclean@franco-little.org,Pachecotown,TX,franco-little.org,Andre Murray,509,C00509,2025-09-03T05:36:15.544Z,2025-09-03T05:36:15.544Z
C00929,buchananjerry@evans-allen.biz,Wellsshire,NC,evans-allen.biz,Timothy Atkinson,929,C00929,2025-09-03T05:36:15.544Z,2025-09-03T05:36:15.544Z
C01462,sflynn@yahoo.com,Justinstad,MD,yahoo.com,Melody Daniels,1462,C01462,2025-09-03T05:36:15.544Z,2025-09-03T05:36:15.544Z
C01720,carolmartin@yahoo.com,North Brianview,IN,yahoo.com,Cindy Gonzalez,1720,C01720,2025-09-03T05:36:15.544Z,2025-09-03T05:36:15.544Z
C01815,aperez@gmail.com,Port Michaelstad,ID,gmail.com,Laura Kramer,1815,C01815,2025-09-03T05:36:15.544Z,2025-09-03T05:36:15.544Z
C00006,traceyramos@gmail.com,North Matthew,IN,gmail.com,Kevin Mccarthy,6,C00006,2025-09-03T05:36:15.544Z,2025-09-03T05:36:15.544Z


**Separating New vs Old Records**

In [0]:
df_new = df_join.filter(df_old.old_DimCustomerKey.isNull())

In [0]:
#Already added in gold

df_old = df_join.filter(df_old.old_DimCustomerKey.isNotNull())

**Preparing df_old**

In [0]:
#Dropping all the columns which are not required
df_old = df_old.drop("old_customer_id", "old_update_date")

#Renaming "old_DimCustomerKey" to "DimCustomerKey"
df_old = df_old.withColumnRenamed("old_DimCustomerKey", "DimCustomerKey")

#Renaming old_create_date column to create_date
df_old = df_old.withColumnRenamed("old_create_date", "create_date")
df_old = df_old.withColumn("create_date", to_timestamp(col("create_date")))

#Recreating update_date column
df_old = df_old.withColumn("update_date", current_timestamp())

In [0]:
df_old.display()

customer_id,email,city,state,domains,full_name,DimCustomerKey,create_date,update_date
C00049,kristinawalsh@gmail.com,New Heatherside,IA,gmail.com,Mike Harvey,49,2025-09-03T05:36:15.544Z,2025-09-03T05:47:03.529Z
C00126,bonniewhite@smith.com,Gregoryton,WA,smith.com,Amanda King,126,2025-09-03T05:36:15.544Z,2025-09-03T05:47:03.529Z
C00169,michaelfox@hotmail.com,Michaelburgh,DC,hotmail.com,Holly Ward,169,2025-09-03T05:36:15.544Z,2025-09-03T05:47:03.529Z
C00394,thomasmontgomery@gonzalez-ross.com,North Denisemouth,NJ,gonzalez-ross.com,Anthony Kelly,394,2025-09-03T05:36:15.544Z,2025-09-03T05:47:03.529Z
C00509,dylanmclean@franco-little.org,Pachecotown,TX,franco-little.org,Andre Murray,509,2025-09-03T05:36:15.544Z,2025-09-03T05:47:03.529Z
C00929,buchananjerry@evans-allen.biz,Wellsshire,NC,evans-allen.biz,Timothy Atkinson,929,2025-09-03T05:36:15.544Z,2025-09-03T05:47:03.529Z
C01462,sflynn@yahoo.com,Justinstad,MD,yahoo.com,Melody Daniels,1462,2025-09-03T05:36:15.544Z,2025-09-03T05:47:03.529Z
C01720,carolmartin@yahoo.com,North Brianview,IN,yahoo.com,Cindy Gonzalez,1720,2025-09-03T05:36:15.544Z,2025-09-03T05:47:03.529Z
C01815,aperez@gmail.com,Port Michaelstad,ID,gmail.com,Laura Kramer,1815,2025-09-03T05:36:15.544Z,2025-09-03T05:47:03.529Z
C00006,traceyramos@gmail.com,North Matthew,IN,gmail.com,Kevin Mccarthy,6,2025-09-03T05:36:15.544Z,2025-09-03T05:47:03.529Z


**Preparing df_new**

In [0]:
df_new.display()

customer_id,email,city,state,domains,full_name,old_DimCustomerKey,old_customer_id,old_create_date,old_update_date


In [0]:
#Dropping all the columns which are not required
df_new = df_new.drop("old_DimCustomerKey", "old_customer_id", "old_update_date", "old_create_date")

#Recreating "update_date", "create_date" columns with current timestamp
df_new = df_new.withColumn("create_date", current_timestamp())
df_new = df_new.withColumn("update_date", current_timestamp())

In [0]:
df_new.display()

customer_id,email,city,state,domains,full_name,create_date,update_date


**Surrogate Key - From 1**

In [0]:
windowSpec = Window.orderBy(col("customer_id"))

**Adding Max Surrogate Key**

In [0]:
if init_load_flag == 1:
    max_surrogate_key = 0
else:
    df_maxsur = spark.sql("select max(DimCustomerKey) as max_surrogate_key from databricks_cata.gold.DimCustomers")
    max_surrogate_key = df_maxsur.collect()[0]['max_surrogate_key']
    if max_surrogate_key is None:
        max_surrogate_key = 0

df_new = df_new.withColumn("DimCustomerKey", lit(max_surrogate_key) + row_number().over(windowSpec))

df_new.display()

/databricks/spark/python/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


customer_id,email,city,state,domains,full_name,create_date,update_date,DimCustomerKey


**Union of df_old and df_new**

In [0]:
df_final = df_new.unionByName(df_old)

/databricks/spark/python/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
df_final.display()

/databricks/spark/python/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/spark/python/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


customer_id,email,city,state,domains,full_name,create_date,update_date,DimCustomerKey
C00049,kristinawalsh@gmail.com,New Heatherside,IA,gmail.com,Mike Harvey,2025-09-03T05:36:15.544Z,2025-09-03T05:49:20.952Z,49
C00126,bonniewhite@smith.com,Gregoryton,WA,smith.com,Amanda King,2025-09-03T05:36:15.544Z,2025-09-03T05:49:20.952Z,126
C00169,michaelfox@hotmail.com,Michaelburgh,DC,hotmail.com,Holly Ward,2025-09-03T05:36:15.544Z,2025-09-03T05:49:20.952Z,169
C00394,thomasmontgomery@gonzalez-ross.com,North Denisemouth,NJ,gonzalez-ross.com,Anthony Kelly,2025-09-03T05:36:15.544Z,2025-09-03T05:49:20.952Z,394
C00509,dylanmclean@franco-little.org,Pachecotown,TX,franco-little.org,Andre Murray,2025-09-03T05:36:15.544Z,2025-09-03T05:49:20.952Z,509
C00929,buchananjerry@evans-allen.biz,Wellsshire,NC,evans-allen.biz,Timothy Atkinson,2025-09-03T05:36:15.544Z,2025-09-03T05:49:20.952Z,929
C01462,sflynn@yahoo.com,Justinstad,MD,yahoo.com,Melody Daniels,2025-09-03T05:36:15.544Z,2025-09-03T05:49:20.952Z,1462
C01720,carolmartin@yahoo.com,North Brianview,IN,yahoo.com,Cindy Gonzalez,2025-09-03T05:36:15.544Z,2025-09-03T05:49:20.952Z,1720
C01815,aperez@gmail.com,Port Michaelstad,ID,gmail.com,Laura Kramer,2025-09-03T05:36:15.544Z,2025-09-03T05:49:20.952Z,1815
C00006,traceyramos@gmail.com,North Matthew,IN,gmail.com,Kevin Mccarthy,2025-09-03T05:36:15.544Z,2025-09-03T05:49:20.952Z,6


##**SCD Type-1**

In [0]:
#Not the part of actual code

if spark.catalog.tableExists("databricks_cata.gold.DimCustomers"):
    print("Hello")
else:
    print("Broo")

Hello


In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists("databricks_cata.gold.DimCustomers"):
    #dlt_obj is the target, our main table where we have to apply our merge condition
    dlt_obj = DeltaTable.forPath(spark, "abfss://gold@databrickscygnusete.dfs.core.windows.net/DimCustomers")

    dlt_obj.alias("trg").merge(df_final.alias("src"), "trg.DimCustomerKey = src.DimCustomerKey").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

else:
    df_final.write.mode("overwrite").format("delta").option("path", "abfss://gold@databrickscygnusete.dfs.core.windows.net/DimCustomers").saveAsTable("databricks_cata.gold.DimCustomers")
 

/databricks/spark/python/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
%sql
SELECT * FROM databricks_cata.gold.dimcustomers

customer_id,email,city,state,domains,full_name,create_date,update_date,DimCustomerKey
C00001,rushjeff@ryan.org,Johnsonmouth,MS,ryan.org,Emily Mooney,2025-09-03T05:36:15.544Z,2025-09-03T05:50:20.684Z,1
C00002,mccoykiara@kelly.com,Stephenfort,WY,kelly.com,Andrea Sellers,2025-09-03T05:36:15.544Z,2025-09-03T05:50:20.684Z,2
C00003,rebeccamiller@yahoo.com,South Stephenshire,LA,yahoo.com,Craig Hayes,2025-09-03T05:36:15.544Z,2025-09-03T05:50:20.684Z,3
C00004,lawrence05@campbell.info,Chrisland,ND,campbell.info,Bryan Scott,2025-09-03T05:36:15.544Z,2025-09-03T05:50:20.684Z,4
C00005,carrie45@yahoo.com,East Dennistown,RI,yahoo.com,Sean Vasquez,2025-09-03T05:36:15.544Z,2025-09-03T05:50:20.684Z,5
C00006,traceyramos@gmail.com,North Matthew,IN,gmail.com,Kevin Mccarthy,2025-09-03T05:36:15.544Z,2025-09-03T05:50:20.684Z,6
C00007,scottallen@gmail.com,Joneshaven,VA,gmail.com,Amanda Doyle,2025-09-03T05:36:15.544Z,2025-09-03T05:50:20.684Z,7
C00008,sullivanjeremy@horton-adams.com,South Nathanfurt,CT,horton-adams.com,Paul Campos,2025-09-03T05:36:15.544Z,2025-09-03T05:50:20.684Z,8
C00009,dennis03@yahoo.com,Kimberlyview,MD,yahoo.com,Mary Green,2025-09-03T05:36:15.544Z,2025-09-03T05:50:20.684Z,9
C00010,charles58@murillo.net,West Hector,OK,murillo.net,James Myers,2025-09-03T05:36:15.544Z,2025-09-03T05:50:20.684Z,10
